# 实践实验：决策树

在本练习中，你将从零开始实现一个决策树，并将其应用于判断蘑菇可食用还是有毒的分类任务。

# 大纲
- [1 - 软件包](#1)
- [2 -  问题描述](#2)
- [3 - 数据集](#3)
  - [3.1 独热编码后的数据集](#3.1)
- [4 - 决策树回顾](#4)
  - [4.1  计算熵](#4.1)
    - [练习 1](#ex01)
  - [4.2  划分数据集](#4.2)
    - [练习 2](#ex02)
  - [4.3  计算信息增益](#4.3)
    - [练习 3](#ex03)
  - [4.4  获取最佳划分](#4.4)
    - [练习 4](#ex04)
- [5 - 构建决策树](#5)

<a name="1"></a>
## 1 - 软件包

首先，运行下面的单元格，导入本次作业所需的全部软件包。
- [numpy](https://www.numpy.org) 是 Python 中处理矩阵的基础软件包。
- [matplotlib](https://matplotlib.org) 是 Python 中著名的绘图库。
- ``utils.py`` 包含本次作业所需的辅助函数。你不需要修改这个文件中的代码。

In [6]:
import numpy as np
import matplotlib.pyplot as plt
from public_tests import *

%matplotlib inline

<a name="2"></a>
## 2 - 问题描述

假设你要创办一家种植和销售野生蘑菇的公司。 
- 由于并非所有蘑菇都可食用，你希望能够根据蘑菇的物理特征判断某个蘑菇是可食用还是有毒的
- 你已经有一些现成的数据可用于这项任务。

你能利用这些数据来帮助识别哪些蘑菇可以安全出售吗？

注意：使用的数据集仅用于演示目的，并非识别可食用蘑菇的指南。



<a name="3"></a>
## 3 - 数据集

你将首先加载这项任务所需的数据集。你收集到的数据集如下：

| 菌盖颜色 | 菌柄形状 | Solitary | 可食用 |
|:---------:|:-----------:|:--------:|:------:|
|   Brown   |   Tapering  |    Yes   |    1   |
|   Brown   |  Enlarging  |    Yes   |    1   |
|   Brown   |  Enlarging  |    No    |    0   |
|   Brown   |  Enlarging  |    No    |    0   |
|   Brown   |   Tapering  |    Yes   |    1   |
|    Red    |   Tapering  |    Yes   |    0   |
|    Red    |  Enlarging  |    No    |    0   |
|   Brown   |  Enlarging  |    Yes   |    1   |
|    Red    |   Tapering  |    No    |    1   |
|   Brown   |  Enlarging  |    No    |    0   |


-  你有 10 个蘑菇样本。对于每个样本，你有
    - 三个特征
        - 菌盖颜色（`Brown` 或 `Red`）、
        - 菌柄形状（`Tapering` 或 `Enlarging`），以及
        - Solitary（`Yes` 或 `No`）
    - 标签
        - 可食用（`1` 表示是，`0` 表示有毒）

<a name="3.1"></a>
### 3.1 独热编码后的数据集
为了便于实现，我们对特征进行了独热编码（将它们转换为取值为 0 或 1 的特征）

| 菌盖为 Brown | 菌柄形状为 Tapering | Solitary | 可食用 |
|:---------:|:--------------------:|:--------:|:------:|
|     1     |           1          |     1    |    1   |
|     1     |           0          |     1    |    1   |
|     1     |           0          |     0    |    0   |
|     1     |           0          |     0    |    0   |
|     1     |           1          |     1    |    1   |
|     0     |           1          |     1    |    0   |
|     0     |           0          |     0    |    0   |
|     1     |           0          |     1    |    1   |
|     0     |           1          |     0    |    1   |
|     1     |           0          |     0    |    0   |

因此，
- `X_train` 包含每个样本的三个特征 
    - Brown 颜色（值为 `1` 表示菌盖颜色为“Brown”，`0` 表示菌盖颜色为“Red”）
    - Tapering 形状（值为 `1` 表示菌柄形状为“Tapering Stalk Shape”，`0` 表示菌柄形状为“Enlarging”）
    - Solitary（值为 `1` 表示“Yes”，`0` 表示“No”）

- `y_train` 表示蘑菇是否可食用 
    - `y = 1` 表示可食用
    - `y = 0` 表示有毒

In [7]:
X_train = np.array([[1,1,1],[1,0,1],[1,0,0],[1,0,0],[1,1,1],[0,1,1],[0,0,0],[1,0,1],[0,1,0],[1,0,0]])
y_train = np.array([1,1,0,0,1,0,0,1,1,0])

#### 查看变量
让我们进一步熟悉你的数据集。  
- 一个很好的起点是直接打印每个变量，看看它包含什么内容。

下面的代码会打印 `X_train` 的前几个元素以及该变量的类型。

In [8]:
print("First few elements of X_train:\n", X_train[:5])
print("Type of X_train:",type(X_train))

First few elements of X_train:
 [[1 1 1]
 [1 0 1]
 [1 0 0]
 [1 0 0]
 [1 1 1]]
Type of X_train: <class 'numpy.ndarray'>


现在，让我们对 `y_train` 做同样的事情

In [9]:
print("First few elements of y_train:", y_train[:5])
print("Type of y_train:",type(y_train))

First few elements of y_train: [1 1 0 0 1]
Type of y_train: <class 'numpy.ndarray'>


#### 检查变量的维度

另一种熟悉数据的有用方法是查看它的维度。

请打印 `X_train` 和 `y_train` 的形状，看看你的数据集中有多少训练样本。

In [10]:
print ('The shape of X_train is:', X_train.shape)
print ('The shape of y_train is: ', y_train.shape)
print ('Number of training examples (m):', len(X_train))

The shape of X_train is: (10, 3)
The shape of y_train is:  (10,)
Number of training examples (m): 10


<a name="4"></a>
## 4 - 决策树回顾

在这个实践实验中，你将基于所提供的数据集构建一个决策树。

- 回顾一下，构建决策树的步骤如下：
    - 从根节点处的全部样本开始
    - 计算按所有可能特征进行划分的信息增益，并选择信息增益最高的那个
    - 按所选特征划分数据集，并创建树的左分支和右分支
    - 不断重复划分过程，直到满足停止条件
  
  
- 在本实验中，你将实现以下几个函数，它们能让你使用信息增益最高的特征把一个节点划分为左分支和右分支
    - 计算某个节点处的熵 
    - 根据给定特征把某个节点处的数据集划分为左分支和右分支
    - 计算按给定特征进行划分所得到的信息增益
    - 选择使信息增益最大化的特征
    
- 然后，我们将使用你实现的这些辅助函数，通过重复划分过程来构建决策树，直到满足停止条件 
    - 在本实验中，我们选择的停止条件是将最大深度设为 2

<a name="4.1"></a>
### 4.1  计算熵

首先，你将编写一个名为 `compute_entropy` 的辅助函数，用于计算某个节点处的熵（不纯度的度量）。 
- 该函数接收一个 numpy 数组（`y`），用于指示该节点中的样本是可食用的（`1`）还是有毒的（`0`） 

完成下面的 `compute_entropy()` 函数，使其：
* 计算 $p_1$，即可食用样本所占的比例（即在 `y` 中取值为 `1` 的样本）
* 然后按如下公式计算熵 

$$H(p_1) = -p_1 \text{log}_2(p_1) - (1- p_1) \text{log}_2(1- p_1)$$
* 注意 
    * 对数以 $2$ 为底计算 
    * 出于实现上的考虑，$0\text{log}_2(0) = 0$。也就是说，如果 `p_1 = 0` 或 `p_1 = 1`，则将熵设为 `0`
    * 务必检查节点处的数据是否为空（即 `len(y) != 0`）。如果为空则返回 `0`
    
<a name="ex01"></a>
### 练习 1

请按照上面的说明完成 `compute_entropy()` 函数。
    
如果遇到困难，可以查看下面单元格之后给出的提示，帮助你完成实现。

In [22]:
# UNQ_C1
# GRADED FUNCTION: compute_entropy

def compute_entropy(y):
    """
    Computes the entropy for 
    
    Args:
       y (ndarray): Numpy array indicating whether each example at a node is
           edible (`1`) or poisonous (`0`)
       
    Returns:
        entropy (float): Entropy at that node
        
    """
    # 你需要正确返回以下变量
    entropy = 0.
    
    ### START CODE HERE ###
    if len(y) != 0:
        p1 = p1 = len(y[y == 1]) / len(y) 
     # 当 p1 为 0 或 1 时，将熵设为 0（以处理 0log0）
        if p1 != 0 and p1 != 1:
             entropy = -p1 * np.log2(p1) - (1 - p1) * np.log2(1 - p1)
        else:
             entropy = 0
    ### END CODE HERE ###        
    
    return entropy

<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    
    
   * 要计算 `p1`
       * 你可以通过 `y[y == 1]` 得到 `y` 中取值为 `1` 的样本子集
       * 你可以使用 `len(y)` 来获取 `y` 中样本的数量
   * 要计算 `entropy`
       * <a href="https://numpy.org/doc/stable/reference/generated/numpy.log2.html">np.log2</a> 可以让你计算 numpy 数组以 2 为底的对数
       * 如果 `p1` 的值为 0 或 1，请务必将熵设为 `0` 
     
    <details>
          <summary><font size="2" color="darkblue"><b> 点击查看更多提示</b></font></summary>
        
    * 下面是这个函数整体实现的组织方式
    ```python 
    def compute_entropy(y):
        
        # You need to return the following variables correctly
        entropy = 0.

        ### START CODE HERE ###
        if len(y) != 0:
            # Your code here to calculate the fraction of edible examples (i.e with value = 1 in y)
            p1 =

            # For p1 = 0 and 1, set the entropy to 0 (to handle 0log0)
            if p1 != 0 and p1 != 1:
                # Your code here to calculate the entropy using the formula provided above
                entropy = 
            else:
                entropy = 0. 
        ### END CODE HERE ###        

        return entropy
    ```
    
    如果你仍然遇到困难，可以查看下面给出的提示，弄清楚如何计算 `p1` 和 `entropy`。
    
    <details>
          <summary><font size="2" color="darkblue"><b>计算 p1 的提示</b></font></summary>
           &emsp; &emsp; 你可以按如下方式计算 p1：<code>p1 = len(y[y == 1]) / len(y) </code>
    </details>

     <details>
          <summary><font size="2" color="darkblue"><b>计算熵的提示</b></font></summary>
          &emsp; &emsp; 你可以按如下方式计算熵：<code>entropy = -p1 * np.log2(p1) - (1 - p1) * np.log2(1 - p1)</code>
    </details>
        
    </details>

</details>

    

你可以运行下面的测试代码来检验你的实现是否正确：

In [23]:
# 计算根节点处的熵（即使用全部样本）
# 由于我们有 5 个可食用和 5 个不可食用的蘑菇，熵应该为 1

print("Entropy at root node: ", compute_entropy(y_train)) 

# UNIT TESTS
compute_entropy_test(compute_entropy)

Entropy at root node:  1.0
 All tests passed.


**期望输出**:
<table>
  <tr>
    <td> <b>根节点处的熵：<b> 1.0 </td> 
  </tr>
</table>

<a name="4.2"></a>
### 4.2  划分数据集

接下来，你将编写一个名为 `split_dataset` 的辅助函数，它接收某个节点处的数据和一个用于划分的特征，并将其划分为左分支和右分支。在本实验的后面，你还会实现用于计算划分优劣的代码。

- 该函数接收训练数据、该节点处数据点的索引列表，以及用于划分的特征。 
- 它对数据进行划分，并返回左分支和右分支上的索引子集。
- 例如，假设我们从根节点开始（即 `node_indices = [0,1,2,3,4,5,6,7,8,9]`），并且选择按特征 `0` 划分，该特征表示样本的菌盖是否为棕色。
    - 那么该函数的输出就是 `left_indices = [0,1,2,3,4,7,9]` 和 `right_indices = [5,6,8]`
    
| 索引 | 菌盖为 Brown | 菌柄形状为 Tapering | Solitary | 可食用 |
|:-----:|:---------:|:--------------------:|:--------:|:------:|
|   0   |     1     |           1          |     1    |    1   |
|   1   |     1     |           0          |     1    |    1   |
|   2   |     1     |           0          |     0    |    0   |
|   3   |     1     |           0          |     0    |    0   |
|   4   |     1     |           1          |     1    |    1   |
|   5   |     0     |           1          |     1    |    0   |
|   6   |     0     |           0          |     0    |    0   |
|   7   |     1     |           0          |     1    |    1   |
|   8   |     0     |           1          |     0    |    1   |
|   9   |     1     |           0          |     0    |    0   |

<a name="ex02"></a>
### 练习 2

请完成下面所示的 `split_dataset()` 函数

- 对于 `node_indices` 中的每个索引
    - 如果该特征在该索引处 `X` 的取值为 `1`，则将该索引添加到 `left_indices`
    - 如果该特征在该索引处 `X` 的取值为 `0`，则将该索引添加到 `right_indices`

如果遇到困难，可以查看下面单元格之后给出的提示，帮助你完成实现。

In [24]:
# UNQ_C2
# GRADED FUNCTION: split_dataset

def split_dataset(X, node_indices, feature):
    """
    Splits the data at the given node into
    left and right branches
    
    Args:
        X (ndarray):             Data matrix of shape(n_samples, n_features)
        node_indices (list):  List containing the active indices. I.e, the samples being considered at this step.
        feature (int):           Index of feature to split on
    
    Returns:
        left_indices (list): Indices with feature value == 1
        right_indices (list): Indices with feature value == 0
    """
    
    # 你需要正确返回以下变量
    left_indices = []
    right_indices = []
    
    ### START CODE HERE ###
    for i in node_indices:   
        if X[i][feature] == 1:
            left_indices.append(i)
        else:
            right_indices.append(i)
    ### END CODE HERE ###
        
    return left_indices, right_indices

<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    
    
   * 下面是这个函数整体实现的组织方式
    ```python 
    def split_dataset(X, node_indices, feature):
    
        # You need to return the following variables correctly
        left_indices = []
        right_indices = []

        ### START CODE HERE ###
        # Go through the indices of examples at that node
        for i in node_indices:   
            if # Your code here to check if the value of X at that index for the feature is 1
                left_indices.append(i)
            else:
                right_indices.append(i)
        ### END CODE HERE ###
        
    return left_indices, right_indices
    ```
    <details>
          <summary><font size="2" color="darkblue"><b> 点击查看更多提示</b></font></summary>
        
    条件是 <code> if X[i][feature] == 1:</code>。
        
    </details>

</details>

    

现在，让我们用下面的代码块检查你的实现。如前所述，让我们尝试在包含全部样本的根节点处按特征 0（菌盖为 Brown）划分数据集

In [25]:
root_indices = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

# 这些变量可以随意调整
# 数据集只有三个特征，因此该值可以是 0（Brown Cap）、1（Tapering Stalk Shape）或 2（Solitary）
feature = 0

left_indices, right_indices = split_dataset(X_train, root_indices, feature)

print("Left indices: ", left_indices)
print("Right indices: ", right_indices)

# UNIT TESTS    
split_dataset_test(split_dataset)

Left indices:  [0, 1, 2, 3, 4, 7, 9]
Right indices:  [5, 6, 8]
 All tests passed.


**期望输出**:
```
Left indices:  [0, 1, 2, 3, 4, 7, 9]
Right indices:  [5, 6, 8]
```

<a name="4.3"></a>
### 4.3  计算信息增益

接下来，你将编写一个名为 `information_gain` 的函数，它接收训练数据、某个节点处的索引和一个用于划分的特征，并返回这次划分所带来的信息增益。

<a name="ex03"></a>
### 练习 3

请完成下面所示的 `compute_information_gain()` 函数，以计算

$$\text{Information Gain} = H(p_1^\text{node})- (w^{\text{left}}H(p_1^\text{left}) + w^{\text{right}}H(p_1^\text{right}))$$

其中 
- $H(p_1^\text{node})$ 是该节点处的熵 
- $H(p_1^\text{left})$ 和 $H(p_1^\text{right})$ 是划分后左分支和右分支处的熵
- $w^{\text{left}}$ 和 $w^{\text{right}}$ 分别是左分支和右分支上样本所占的比例

注意：
- 你可以使用上面实现的 `compute_entropy()` 函数来计算熵
- 我们已经提供了一些初始代码，它们使用你上面实现的 `split_dataset()` 函数来划分数据集 

如果遇到困难，可以查看下面单元格之后给出的提示，帮助你完成实现。

In [26]:
# UNQ_C3
# GRADED FUNCTION: compute_information_gain

def compute_information_gain(X, y, node_indices, feature):
    
    """
    Compute the information of splitting the node on a given feature
    
    Args:
        X (ndarray):            Data matrix of shape(n_samples, n_features)
        y (array like):         list or ndarray with n_samples containing the target variable
        node_indices (ndarray): List containing the active indices. I.e, the samples being considered in this step.
   
    Returns:
        cost (float):        Cost computed
    
    """    
    # 划分数据集
    left_indices, right_indices = split_dataset(X, node_indices, feature)
    
    # 一些有用的变量
    X_node, y_node = X[node_indices], y[node_indices]
    X_left, y_left = X[left_indices], y[left_indices]
    X_right, y_right = X[right_indices], y[right_indices]
    
    # 你需要正确返回以下变量
    information_gain = 0
    
    ### START CODE HERE ###
    node_entropy = compute_entropy(y_node)
    left_entropy = compute_entropy(y_left)
    right_entropy = compute_entropy(y_right)
    
    # 权重 
    w_left = len(X_left) / len(X_node)
    w_right = len(X_right) / len(X_node)
    
    #加权熵
    weighted_entropy = w_left * left_entropy + w_right * right_entropy
    
    #信息增益 
    information_gain = node_entropy - weighted_entropy
    
    ### END CODE HERE ###  
    
    return information_gain

<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    
    
   * 下面是这个函数整体实现的组织方式
    ```python 
    def compute_information_gain(X, y, node_indices, feature):
        # Split dataset
        left_indices, right_indices = split_dataset(X, node_indices, feature)

        # Some useful variables
        X_node, y_node = X[node_indices], y[node_indices]
        X_left, y_left = X[left_indices], y[left_indices]
        X_right, y_right = X[right_indices], y[right_indices]

        # You need to return the following variables correctly
        information_gain = 0

        ### START CODE HERE ###
        # Your code here to compute the entropy at the node using compute_entropy()
        node_entropy = 
        # Your code here to compute the entropy at the left branch
        left_entropy = 
        # Your code here to compute the entropy at the right branch
        right_entropy = 

        # Your code here to compute the proportion of examples at the left branch
        w_left = 
        
        # Your code here to compute the proportion of examples at the right branch
        w_right = 

        # Your code here to compute weighted entropy from the split using 
        # w_left, w_right, left_entropy and right_entropy
        weighted_entropy = 

        # Your code here to compute the information gain as the entropy at the node
        # minus the weighted entropy
        information_gain = 
        ### END CODE HERE ###  

        return information_gain
    ```
    如果你仍然遇到困难，请查看下面的提示。
    
    <details>
          <summary><font size="2" color="darkblue"><b> 计算各熵值的提示</b></font></summary>
        
    <code>node_entropy = compute_entropy(y_node)</code><br>
    <code>left_entropy = compute_entropy(y_left)</code><br>
    <code>right_entropy = compute_entropy(y_right)</code>
        
    </details>
    
    <details>
          <summary><font size="2" color="darkblue"><b>计算 w_left 和 w_right 的提示</b></font></summary>
           <code>w_left = len(X_left) / len(X_node)</code><br>
           <code>w_right = len(X_right) / len(X_node)</code>
    </details>
    
    <details>
          <summary><font size="2" color="darkblue"><b>计算 weighted_entropy 的提示</b></font></summary>
           <code>weighted_entropy = w_left * left_entropy + w_right * right_entropy</code>
    </details>
    
    <details>
          <summary><font size="2" color="darkblue"><b>计算 information_gain 的提示</b></font></summary>
           <code> information_gain = node_entropy - weighted_entropy</code>
    </details>


</details>

你现在可以使用下面的单元格检查你的实现，并计算按每个特征进行划分所能带来的信息增益

In [27]:
info_gain0 = compute_information_gain(X_train, y_train, root_indices, feature=0)
print("Information Gain from splitting the root on brown cap: ", info_gain0)
    
info_gain1 = compute_information_gain(X_train, y_train, root_indices, feature=1)
print("Information Gain from splitting the root on tapering stalk shape: ", info_gain1)

info_gain2 = compute_information_gain(X_train, y_train, root_indices, feature=2)
print("Information Gain from splitting the root on solitary: ", info_gain2)

# UNIT TESTS
compute_information_gain_test(compute_information_gain)

Information Gain from splitting the root on brown cap:  0.034851554559677034
Information Gain from splitting the root on tapering stalk shape:  0.12451124978365313
Information Gain from splitting the root on solitary:  0.2780719051126377
 All tests passed.


**期望输出**:
```
Information Gain from splitting the root on brown cap:  0.034851554559677034
Information Gain from splitting the root on tapering stalk shape:  0.12451124978365313
Information Gain from splitting the root on solitary:  0.2780719051126377
```

在根节点处按“Solitary”（feature = 2）进行划分可以获得最大的信息增益。因此，它是根节点处最佳的特征。

<a name="4.4"></a>
### 4.4  获取最佳划分
现在让我们编写一个函数来获取最佳划分特征：像上面那样计算每个特征的信息增益，并返回信息增益最大的特征

<a name="ex04"></a>
### 练习 4
请完成下面所示的 `get_best_split()` 函数。
- 该函数接收训练数据以及该节点处数据点的索引
- 函数的输出是信息增益最大的特征 
    - 你可以使用 `compute_information_gain()` 函数遍历各个特征并计算每个特征的信息增益
如果遇到困难，可以查看下面单元格之后给出的提示，帮助你完成实现。

In [36]:
# UNQ_C4
# GRADED FUNCTION: get_best_split

def get_best_split(X, y, node_indices):   
    """
    Returns the optimal feature and threshold value
    to split the node data 
    
    Args:
        X (ndarray):            Data matrix of shape(n_samples, n_features)
        y (array like):         list or ndarray with n_samples containing the target variable
        node_indices (ndarray): List containing the active indices. I.e, the samples being considered in this step.

    Returns:
        best_feature (int):     The index of the best feature to split
    """    
    
    # 一些有用的变量
    num_features = X.shape[1]
    
    # 你需要正确返回以下变量
    best_feature = -1
    
    ### START CODE HERE ###
    max_info_gain=0
    for feature in range(num_features):
        info_gain = compute_information_gain(X, y, node_indices, feature)
        if info_gain > max_info_gain:
            max_info_gain = info_gain
            best_feature = feature
                
        
    ### END CODE HERE ##    
       
    return best_feature


<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    
    
   * 下面是这个函数整体实现的组织方式
    
    ```python 
    def get_best_split(X, y, node_indices):   

        # Some useful variables
        num_features = X.shape[1]

        # You need to return the following variables correctly
        best_feature = -1

        ### START CODE HERE ###
        max_info_gain = 0

        # Iterate through all features
        for feature in range(num_features): 
            
            # Your code here to compute the information gain from splitting on this feature
            info_gain = 
            
            # If the information gain is larger than the max seen so far
            if info_gain > max_info_gain:  
                # Your code here to set the max_info_gain and best_feature
                max_info_gain = 
                best_feature = 
        ### END CODE HERE ##    
   
    return best_feature
    ```
    如果你仍然遇到困难，请查看下面的提示。
    
    <details>
          <summary><font size="2" color="darkblue"><b>计算 info_gain 的提示</b></font></summary>
        
    <code>info_gain = compute_information_gain(X, y, node_indices, feature)</code>
    </details>
    
    <details>
          <summary><font size="2" color="darkblue"><b>更新 max_info_gain 和 best_feature 的提示</b></font></summary>
           <code>max_info_gain = info_gain</code><br>
           <code>best_feature = feature</code>
    </details>
</details>

现在，让我们用下面的单元格检查你的函数实现。

In [37]:
best_feature = get_best_split(X_train, y_train, root_indices)
print("Best feature to split on: %d" % best_feature)

# UNIT TESTS
get_best_split_test(get_best_split)

Best feature to split on: 2
 All tests passed.


如我们在上面看到的，函数返回的结果表明，在根节点处最佳的特征是特征 2（“Solitary”）

<a name="5"></a>
## 5 - 构建决策树

在本节中，我们使用你在上面实现的函数来生成一个决策树：不断选取最佳的特征进行划分，直到达到停止条件（最大深度为 2）。

这部分你不需要实现任何内容。

In [38]:
# 不计分
tree = []

def build_tree_recursive(X, y, node_indices, branch_name, max_depth, current_depth):
    """
    Build a tree using the recursive algorithm that split the dataset into 2 subgroups at each node.
    This function just prints the tree.
    
    Args:
        X (ndarray):            Data matrix of shape(n_samples, n_features)
        y (array like):         list or ndarray with n_samples containing the target variable
        node_indices (ndarray): List containing the active indices. I.e, the samples being considered in this step.
        branch_name (string):   Name of the branch. ['Root', 'Left', 'Right']
        max_depth (int):        Max depth of the resulting tree. 
        current_depth (int):    Current depth. Parameter used during recursive call.
   
    """ 

    # 已达最大深度 - 停止划分
    if current_depth == max_depth:
        formatting = " "*current_depth + "-"*current_depth
        print(formatting, "%s leaf node with indices" % branch_name, node_indices)
        return
   
    # 否则，获取最佳划分并划分数据
    # 获取该节点处的最佳特征和阈值
    best_feature = get_best_split(X, y, node_indices) 
    tree.append((current_depth, branch_name, best_feature, node_indices))
    
    formatting = "-"*current_depth
    print("%s Depth %d, %s: Split on feature: %d" % (formatting, current_depth, branch_name, best_feature))
    
    # 按最佳特征划分数据集
    left_indices, right_indices = split_dataset(X, node_indices, best_feature)
    
    # 继续划分左子节点和右子节点。当前深度加 1
    build_tree_recursive(X, y, left_indices, "Left", max_depth, current_depth+1)
    build_tree_recursive(X, y, right_indices, "Right", max_depth, current_depth+1)


In [39]:
build_tree_recursive(X_train, y_train, root_indices, "Root", max_depth=2, current_depth=0)

 Depth 0, Root: Split on feature: 2
- Depth 1, Left: Split on feature: 0
  -- Left leaf node with indices [0, 1, 4, 7]
  -- Right leaf node with indices [5]
- Depth 1, Right: Split on feature: 1
  -- Left leaf node with indices [8]
  -- Right leaf node with indices [2, 3, 6, 9]
